In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import Image

In [ ]:
# Datos
df = pd.read_csv(r"C:\Users\Mariana\Desktop\mapa_superficial\analog_features_all.csv")

# Coordenadas y valores
points = df[['longitude', 'latitude']].values
values = df['deep_SEC'].values

# Interpolación IDW
def idw(x, y, z, xi, yi, power=2):
    dist = np.sqrt((xi - x[:, None, None])**2 + (yi - y[:, None, None])**2)
    weights = 1.0 / (dist**power + 1e-12)
    return np.sum(weights * z[:, None, None], axis=0) / np.sum(weights, axis=0)

# Crear malla para interpolar
lon_min, lon_max = points[:, 0].min(), points[:, 0].max()
lat_min, lat_max = points[:, 1].min(), points[:, 1].max()
grid_lon, grid_lat = np.meshgrid(
    np.linspace(lon_min, lon_max, 300),
    np.linspace(lat_min, lat_max, 300)
)
interpolated = idw(points[:, 0], points[:, 1], values, grid_lon, grid_lat)

# Limitar colorbar al percentil 95
vmax_95 = np.percentile(values, 95)

# Pozos a etiquetar con valores
highlight_wells = ['AW2', 'AW5', 'AW6', 'AW7', 'BW3', 'LRS69', 'LRS70']

# Dibujar mapa
fig, ax = plt.subplots(figsize=(10, 8))
c = ax.imshow(interpolated, extent=(lon_min, lon_max, lat_min, lat_max),
              origin='lower', cmap='rainbow', aspect='auto',
              vmin=values.min(), vmax=vmax_95)

# Dibujar puntos y etiquetas
for _, row in df.iterrows():
    is_trench = row['name'].lower().startswith('trench') or row['type'].lower() == 'trench'
    marker = 'x' if is_trench else 'o'
    size = 6 if is_trench else 10

    ax.plot(row['longitude'], row['latitude'], marker=marker, markersize=size,
            color='black', linestyle='None')

    if is_trench:
        label = row['name'].split('_')[-1]
        ax.text(row['longitude'], row['latitude'], label, fontsize=8,
                ha='left', va='bottom')
    else:
        label_text = row['name']
        if row['name'] in highlight_wells:
            label_text += f" ({row['deep_SEC']:.0f} µS/cm)"
        ha = 'right' if row['name'] == 'LRS70' else 'left'
        va = 'top' if row['name'] == 'BW3' else 'bottom'
        ax.text(row['longitude'], row['latitude'], label_text, fontsize=10,
                fontweight='bold', ha=ha, va=va)

# Colorbar
cb = fig.colorbar(c, ax=ax)
cb.set_label("deep_SEC (µS/cm) – capped at 95th percentile (~800 µS/cm)")
cb.ax.text(1.05, 1.05, 'Values >800 µS/cm\nnot shown to scale', fontsize=9,
           ha='left', va='bottom', transform=cb.ax.transAxes)

# Leyenda
legend_elements = [
    Line2D([0], [0], marker='x', color='b', label='Trench',
           markeredgecolor='black'),
    Line2D([0], [0], marker='o', color='w', label='Well',
           markeredgecolor='black')
]
ax.legend(handles=legend_elements, loc='upper right')

# Etiquetas y título
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Superficial SEC (µS/cm) (Solinst logger - Ashton Survey)")

plt.tight_layout()
plt.show()


In [ ]:
Image(r"C:\Users\Mariana\Desktop\mapa_superficial\area.png")